In [1]:
#  Import Libraries

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier, XGBRegressor

print("All libraries imported")

All libraries imported


In [2]:
#  Load Data

df = pd.read_csv(r"C:\weather-ai-project\data\weather_preprocessed.csv")

print("Shape :", df.shape)
print("Nulls :", df.isnull().sum().sum())
print("Columns :", df.columns.tolist())

Shape : (142193, 29)
Nulls : 0
Columns : ['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow', 'Year', 'Month', 'Season', 'Temp_Range', 'Humidity_Drop', 'Pressure_Drop', 'Wind_Change']


In [3]:
#  Define X and y

X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

print("X shape :", X.shape)
print("y shape :", y.shape)
print("Target values :", y.value_counts().to_dict())

X shape : (142193, 28)
y shape : (142193,)
Target values : {0: 110316, 1: 31877}


In [4]:
#  Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.value_counts().to_dict())
print("y_test  :", y_test.value_counts().to_dict())

X_train : (113754, 28)
X_test  : (28439, 28)
y_train : {0: 88252, 1: 25502}
y_test  : {0: 22064, 1: 6375}


In [5]:
#  Logistic Regression

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy :", round(accuracy_score(y_test, lr_pred) * 100, 2), "%")
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy : 78.78 %
              precision    recall  f1-score   support

           0       0.93      0.79      0.85     22064
           1       0.52      0.78      0.62      6375

    accuracy                           0.79     28439
   macro avg       0.72      0.79      0.74     28439
weighted avg       0.83      0.79      0.80     28439



In [6]:
#  Decision Tree

dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy :", round(accuracy_score(y_test, dt_pred) * 100, 2), "%")
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy : 78.99 %
              precision    recall  f1-score   support

           0       0.86      0.87      0.87     22064
           1       0.53      0.52      0.52      6375

    accuracy                           0.79     28439
   macro avg       0.70      0.69      0.70     28439
weighted avg       0.79      0.79      0.79     28439



In [7]:
#  Random Forest Classifier

rf = RandomForestClassifier(class_weight="balanced", n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest Accuracy :", round(accuracy_score(y_test, rf_pred) * 100, 2), "%")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy : 85.31 %
              precision    recall  f1-score   support

           0       0.86      0.96      0.91     22064
           1       0.79      0.47      0.59      6375

    accuracy                           0.85     28439
   macro avg       0.83      0.72      0.75     28439
weighted avg       0.85      0.85      0.84     28439



In [8]:
#  XGBoost Classifier

xgb = XGBClassifier(
    scale_pos_weight=3.46,
    n_estimators=100,
    random_state=42,
    eval_metric="logloss"
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

print("XGBoost Accuracy :", round(accuracy_score(y_test, xgb_pred) * 100, 2), "%")
print(classification_report(y_test, xgb_pred))

XGBoost Accuracy : 82.3 %
              precision    recall  f1-score   support

           0       0.93      0.83      0.88     22064
           1       0.58      0.78      0.66      6375

    accuracy                           0.82     28439
   macro avg       0.75      0.81      0.77     28439
weighted avg       0.85      0.82      0.83     28439



In [9]:
#  Classification Summary

summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy": [78.78, 78.99, 85.31, 82.30]
})

print(summary)
print("\nBest Classification Model :", summary.loc[summary["Accuracy"].idxmax(), "Model"])

                 Model  Accuracy
0  Logistic Regression     78.78
1        Decision Tree     78.99
2        Random Forest     85.31
3              XGBoost     82.30

Best Classification Model : Random Forest


In [10]:
# Linear Regression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_pred = lin_reg.predict(X_test)

mae = np.mean(np.abs(y_test - lin_pred))
rmse = np.sqrt(np.mean((y_test - lin_pred) ** 2))
r2 = 1 - (np.sum((y_test - lin_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2))

print("Linear Regression")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

Linear Regression
MAE  : 0.2629
RMSE : 0.3432
R2   : 0.3228


In [11]:
#  Random Forest Regressor

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
rf_reg_pred = rf_reg.predict(X_test)

mae = np.mean(np.abs(y_test - rf_reg_pred))
rmse = np.sqrt(np.mean((y_test - rf_reg_pred) ** 2))
r2 = 1 - (np.sum((y_test - rf_reg_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2))

print("Random Forest Regressor")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

Random Forest Regressor
MAE  : 0.2193
RMSE : 0.3247
R2   : 0.3939


In [12]:
#  XGBoost Regressor

xgb_reg = XGBRegressor(n_estimators=100, random_state=42, eval_metric="rmse")
xgb_reg.fit(X_train, y_train)
xgb_reg_pred = xgb_reg.predict(X_test)

mae = np.mean(np.abs(y_test - xgb_reg_pred))
rmse = np.sqrt(np.mean((y_test - xgb_reg_pred) ** 2))
r2 = 1 - (np.sum((y_test - xgb_reg_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2))

print("XGBoost Regressor")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

XGBoost Regressor
MAE  : 0.2112
RMSE : 0.3194
R2   : 0.4135


In [13]:
#  Regression Summary

reg_summary = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor", "XGBoost Regressor"],
    "MAE":   [0.2629, 0.2193, 0.2112],
    "RMSE":  [0.3432, 0.3247, 0.3194],
    "R2":    [0.3228, 0.3939, 0.4135]
})

print(reg_summary)
print("\nBest Regression Model :", reg_summary.loc[reg_summary["R2"].idxmax(), "Model"])

                     Model     MAE    RMSE      R2
0        Linear Regression  0.2629  0.3432  0.3228
1  Random Forest Regressor  0.2193  0.3247  0.3939
2        XGBoost Regressor  0.2112  0.3194  0.4135

Best Regression Model : XGBoost Regressor


In [14]:
# Save Best Models

joblib.dump(rf, r"C:\weather-ai-project\models\rain_classifier.pkl")
joblib.dump(xgb_reg, r"C:\weather-ai-project\models\temp_regressor.pkl")

print("rain_classifier.pkl  saved")
print("temp_regressor.pkl   saved")

rain_classifier.pkl  saved
temp_regressor.pkl   saved


In [15]:
# Replace rain_classifier with XGBoost Classifier

joblib.dump(xgb, r"C:\weather-ai-project\models\rain_classifier.pkl")

print("rain_classifier.pkl replaced with XGBoost Classifier")

rain_classifier.pkl replaced with XGBoost Classifier


In [16]:
# Verify Saved Models

import os

models = [
    r"C:\weather-ai-project\models\rain_classifier.pkl",
    r"C:\weather-ai-project\models\temp_regressor.pkl",
    r"C:\weather-ai-project\models\scaler.pkl"
]

for path in models:
    status = "Found" if os.path.exists(path) else "Missing"
    print(status, "→", path)

Found → C:\weather-ai-project\models\rain_classifier.pkl
Found → C:\weather-ai-project\models\temp_regressor.pkl
Found → C:\weather-ai-project\models\scaler.pkl


In [17]:
# Install LightGBM

import subprocess
subprocess.run(["pip", "install", "lightgbm"], capture_output=True)

import lightgbm as lgb
print("LightGBM version :", lgb.__version__)

LightGBM version : 4.6.0


In [18]:
# Add LightGBM to Imports

from lightgbm import LGBMClassifier

print("LGBMClassifier imported")

LGBMClassifier imported


In [19]:
# Retrain Regression Models with MaxTemp Target

X_reg = df.drop(columns=["MaxTemp", "RainTomorrow"])
y_reg = df["MaxTemp"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Regression Target : MaxTemp")
print("X_train_r shape   :", X_train_r.shape)
print("X_test_r shape    :", X_test_r.shape)
print("MaxTemp range     :", round(y_reg.min(), 2), "to", round(y_reg.max(), 2))

Regression Target : MaxTemp
X_train_r shape   : (113754, 27)
X_test_r shape    : (28439, 27)
MaxTemp range     : -2.93 to 2.88


In [20]:
# Linear Regression on MaxTemp

lin_reg2 = LinearRegression()
lin_reg2.fit(X_train_r, y_train_r)
lin_pred2 = lin_reg2.predict(X_test_r)

mae  = np.mean(np.abs(y_test_r - lin_pred2))
rmse = np.sqrt(np.mean((y_test_r - lin_pred2) ** 2))
r2   = 1 - (np.sum((y_test_r - lin_pred2) ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2))

print("Linear Regression — MaxTemp")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

Linear Regression — MaxTemp
MAE  : 0.0
RMSE : 0.0
R2   : 1.0


In [21]:
# Fix Data Leakage — Drop Correlated Columns

X_reg = df.drop(columns=[
    "MaxTemp", "RainTomorrow",
    "Temp9am", "Temp3pm",
    "MinTemp", "Temp_Range"
])

y_reg = df["MaxTemp"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Leakage columns dropped")
print("X_train_r shape :", X_train_r.shape)
print("X_test_r shape  :", X_test_r.shape)

Leakage columns dropped
X_train_r shape : (113754, 23)
X_test_r shape  : (28439, 23)


In [22]:
# Linear Regression on MaxTemp — Fixed

lin_reg2 = LinearRegression()
lin_reg2.fit(X_train_r, y_train_r)
lin_pred2 = lin_reg2.predict(X_test_r)

mae  = np.mean(np.abs(y_test_r - lin_pred2))
rmse = np.sqrt(np.mean((y_test_r - lin_pred2) ** 2))
r2   = 1 - (np.sum((y_test_r - lin_pred2) ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2))

print("Linear Regression — MaxTemp Fixed")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

Linear Regression — MaxTemp Fixed
MAE  : 0.5077
RMSE : 0.6457
R2   : 0.5833


In [23]:
# Random Forest Regressor on MaxTemp

rf_reg2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg2.fit(X_train_r, y_train_r)
rf_reg_pred2 = rf_reg2.predict(X_test_r)

mae  = np.mean(np.abs(y_test_r - rf_reg_pred2))
rmse = np.sqrt(np.mean((y_test_r - rf_reg_pred2) ** 2))
r2   = 1 - (np.sum((y_test_r - rf_reg_pred2) ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2))

print("Random Forest Regressor — MaxTemp")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

Random Forest Regressor — MaxTemp
MAE  : 0.2709
RMSE : 0.3589
R2   : 0.8713


In [24]:
# XGBoost Regressor on MaxTemp

xgb_reg2 = XGBRegressor(n_estimators=100, random_state=42, eval_metric="rmse")
xgb_reg2.fit(X_train_r, y_train_r)
xgb_reg_pred2 = xgb_reg2.predict(X_test_r)

mae  = np.mean(np.abs(y_test_r - xgb_reg_pred2))
rmse = np.sqrt(np.mean((y_test_r - xgb_reg_pred2) ** 2))
r2   = 1 - (np.sum((y_test_r - xgb_reg_pred2) ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2))

print("XGBoost Regressor — MaxTemp")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

XGBoost Regressor — MaxTemp
MAE  : 0.2513
RMSE : 0.3299
R2   : 0.8913


In [25]:
# LightGBM Classifier

lgbm = LGBMClassifier(class_weight="balanced", n_estimators=100, random_state=42)
lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_test)

print("LightGBM Accuracy :", round(accuracy_score(y_test, lgbm_pred) * 100, 2), "%")
print(classification_report(y_test, lgbm_pred))

[LightGBM] [Info] Number of positive: 25502, number of negative: 88252
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3062
[LightGBM] [Info] Number of data points in the train set: 113754, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM Accuracy : 81.04 %
              precision    recall  f1-score   support

           0       0.93      0.81      0.87     22064
           1       0.55      0.80      0.66      6375

    accuracy                           0.81     28439
   macro avg       0.74      0.81      0.76     28439
weighted avg       0.85      0.81      0.82     28439



In [26]:
# Final Classification Summary

clf_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", 
              "Random Forest", "XGBoost", "LightGBM"],
    "Accuracy": [78.78, 78.99, 85.31, 82.30, 81.04]
})

print(clf_summary)
print("\nBest Accuracy        :", clf_summary.loc[clf_summary["Accuracy"].idxmax(), "Model"])
print("Best for Deployment  : XGBoost → small size, 82.30%")

                 Model  Accuracy
0  Logistic Regression     78.78
1        Decision Tree     78.99
2        Random Forest     85.31
3              XGBoost     82.30
4             LightGBM     81.04

Best Accuracy        : Random Forest
Best for Deployment  : XGBoost → small size, 82.30%


In [27]:
# Final Regression Summary

reg_summary2 = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor", "XGBoost Regressor"],
    "MAE":   [0.5077, 0.2709, 0.2513],
    "RMSE":  [0.6457, 0.3589, 0.3299],
    "R2":    [0.5833, 0.8713, 0.8913]
})

print(reg_summary2)
print("\nBest Regression Model :", reg_summary2.loc[reg_summary2["R2"].idxmax(), "Model"])

                     Model     MAE    RMSE      R2
0        Linear Regression  0.5077  0.6457  0.5833
1  Random Forest Regressor  0.2709  0.3589  0.8713
2        XGBoost Regressor  0.2513  0.3299  0.8913

Best Regression Model : XGBoost Regressor


In [28]:
# Save Final Best Models

joblib.dump(xgb, r"C:\weather-ai-project\models\rain_classifier.pkl")
joblib.dump(xgb_reg2, r"C:\weather-ai-project\models\temp_regressor.pkl")

print("rain_classifier.pkl → XGBoost Classifier saved")
print("temp_regressor.pkl  → XGBoost Regressor saved")

rain_classifier.pkl → XGBoost Classifier saved
temp_regressor.pkl  → XGBoost Regressor saved


In [29]:
# Verify All Models

import os

models = [
    r"C:\weather-ai-project\models\rain_classifier.pkl",
    r"C:\weather-ai-project\models\temp_regressor.pkl",
    r"C:\weather-ai-project\models\scaler.pkl"
]

for path in models:
    size   = round(os.path.getsize(path) / (1024 * 1024), 2)
    status = "Found" if os.path.exists(path) else "Missing"
    print(status, "→", size, "MB →", path)

Found → 0.43 MB → C:\weather-ai-project\models\rain_classifier.pkl
Found → 0.47 MB → C:\weather-ai-project\models\temp_regressor.pkl
Found → 0.0 MB → C:\weather-ai-project\models\scaler.pkl


In [30]:
# LightGBM Regressor on MaxTemp

from lightgbm import LGBMRegressor

lgbm_reg = LGBMRegressor(n_estimators=100, random_state=42)
lgbm_reg.fit(X_train_r, y_train_r)
lgbm_reg_pred = lgbm_reg.predict(X_test_r)

mae  = np.mean(np.abs(y_test_r - lgbm_reg_pred))
rmse = np.sqrt(np.mean((y_test_r - lgbm_reg_pred) ** 2))
r2   = 1 - (np.sum((y_test_r - lgbm_reg_pred) ** 2) / np.sum((y_test_r - y_test_r.mean()) ** 2))

print("LightGBM Regressor — MaxTemp")
print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1788
[LightGBM] [Info] Number of data points in the train set: 113754, number of used features: 23
[LightGBM] [Info] Start training from score -0.000555
LightGBM Regressor — MaxTemp
MAE  : 0.2736
RMSE : 0.3561
R2   : 0.8733


In [31]:
# Updated Final Regression Summary

reg_summary3 = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor", 
              "XGBoost Regressor", "LightGBM Regressor"],
    "MAE":   [0.5077, 0.2709, 0.2513, 0.2736],
    "RMSE":  [0.6457, 0.3589, 0.3299, 0.3561],
    "R2":    [0.5833, 0.8713, 0.8913, 0.8733]
})

print(reg_summary3)
print("\nBest Regression Model :", reg_summary3.loc[reg_summary3["R2"].idxmax(), "Model"])

                     Model     MAE    RMSE      R2
0        Linear Regression  0.5077  0.6457  0.5833
1  Random Forest Regressor  0.2709  0.3589  0.8713
2        XGBoost Regressor  0.2513  0.3299  0.8913
3       LightGBM Regressor  0.2736  0.3561  0.8733

Best Regression Model : XGBoost Regressor


In [32]:
# Feature Importance — XGBoost Classifier

feat_imp_clf = pd.DataFrame({
    "Feature"   : X.columns,
    "Importance": xgb.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 10 Features — Rain Prediction")
print(feat_imp_clf.head(10).to_string(index=False))

Top 10 Features — Rain Prediction
      Feature  Importance
  Humidity3pm    0.281544
     Sunshine    0.105653
     Rainfall    0.072848
WindGustSpeed    0.065333
     Cloud3pm    0.059110
  Pressure3pm    0.054208
     Location    0.024171
Pressure_Drop    0.023424
   WindDir3pm    0.020613
   WindDir9am    0.018785


In [33]:
# Feature Importance — XGBoost Regressor

feat_imp_reg = pd.DataFrame({
    "Feature"   : X_reg.columns,
    "Importance": xgb_reg2.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 10 Features — Temperature Prediction")
print(feat_imp_reg.head(10).to_string(index=False))

Top 10 Features — Temperature Prediction
      Feature  Importance
       Season    0.437438
  Humidity3pm    0.120268
  Evaporation    0.112662
        Month    0.069484
  Pressure3pm    0.049697
Pressure_Drop    0.049387
     Location    0.031713
  Humidity9am    0.017387
   WindDir9am    0.017265
     Sunshine    0.016079


In [34]:
# Save Regression Column List — Critical for Deployment

reg_columns = X_reg.columns.tolist()

import json
with open(r"C:\weather-ai-project\models\reg_columns.json", "w") as f:
    json.dump(reg_columns, f)

print("Regression columns saved → reg_columns.json")
print("Total columns :", len(reg_columns))
print("Columns :", reg_columns)

Regression columns saved → reg_columns.json
Total columns : 23
Columns : ['Location', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'RainToday', 'Year', 'Month', 'Season', 'Humidity_Drop', 'Pressure_Drop', 'Wind_Change']


In [35]:
# Verify All Final Saved Files

import os

files = [
    r"C:\weather-ai-project\models\rain_classifier.pkl",
    r"C:\weather-ai-project\models\temp_regressor.pkl",
    r"C:\weather-ai-project\models\scaler.pkl",
    r"C:\weather-ai-project\models\reg_columns.json"
]

for path in files:
    size   = round(os.path.getsize(path) / (1024 * 1024), 2)
    status = "Found" if os.path.exists(path) else "Missing"
    print(status, "→", size, "MB →", os.path.basename(path))

Found → 0.43 MB → rain_classifier.pkl
Found → 0.47 MB → temp_regressor.pkl
Found → 0.0 MB → scaler.pkl
Found → 0.0 MB → reg_columns.json
